# Single-threshold directional change

This first notebook builds the event ledger for one directional-change threshold. It deliberately stops before plotting, EDA, or regime modelling.

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [27]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from dc_basics.core import log_r, parse_dc_events
from dc_basics.hmm import DistType, InferenceMode, create_simple_hmm_instance
from dc_basics.plotting import (
    plot_dc_indicator_distributions,
    plot_directional_change,
    plot_hmm_event_time_strip,
    plot_hmm_log_r_distributions,
    plot_hmm_next_event_outcomes,
    plot_hmm_next_state_calibration,
    plot_hmm_probability_vs_log_r,
    plot_hmm_recent_event_casebook,
    plot_hmm_states_on_price,
    plot_hmm_tmv_t_space,
)

## Configuration

In [28]:
DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "EURUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.00112
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: EURUSD.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\EURUSD.r.parquet
Threshold: 0.1120%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [29]:
raw = pd.read_parquet(DATA_PATH)

raw = raw.sort_index()
prices = raw["close"].astype(np.float64)

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 240,967
Start: 2023-05-22 14:00:00+02:00
End: 2026-08-13 13:25:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-08-13 13:05:00+02:00,1.15229,1.15249,1.15226,1.15248,163.0,6
2026-08-13 13:10:00+02:00,1.15247,1.15260,1.15241,1.15253,172.0,6
2026-08-13 13:15:00+02:00,1.15252,1.15253,1.15236,1.15242,102.0,6
2026-08-13 13:20:00+02:00,1.15243,1.15247,1.15236,1.15238,129.0,6
2026-08-13 13:25:00+02:00,1.15239,1.15252,1.15234,1.15247,120.0,6


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `event_id` gives each event a stable notebook identifier. `information_time` is deliberately copied from `dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable.

In [30]:
events = parse_dc_events(prices=prices, theta=THETA).copy()

if events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))
events.insert(1, "threshold", THETA)
events.insert(2, "information_time", events["dcc_idx"])

event_columns = [
    "event_id", "threshold", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 6,060


,event_id,threshold,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
6050,6051,0.00112,up,2026-08-06 21:15:00+02:00,1.15155,2026-08-07 17:45:00+02:00,1.15789,2026-08-07 18:30:00+02:00,1.15635,2026-08-07 18:30:00+02:00,4.915735,245,0.000022
6051,6052,0.00112,down,2026-08-07 17:45:00+02:00,1.15789,2026-08-07 20:00:00+02:00,1.15507,2026-08-07 21:10:00+02:00,1.15650,2026-08-07 21:10:00+02:00,2.174522,27,0.000090
6052,6053,0.00112,up,2026-08-07 20:00:00+02:00,1.15507,2026-08-08 00:25:00+02:00,1.15693,2026-08-10 03:10:00+02:00,1.15563,2026-08-10 03:10:00+02:00,1.437761,53,0.000030
6053,6054,0.00112,down,2026-08-08 00:25:00+02:00,1.15693,2026-08-10 09:05:00+02:00,1.15484,2026-08-10 12:40:00+02:00,1.15630,2026-08-10 12:40:00+02:00,1.612951,104,0.000017
6054,6055,0.00112,up,2026-08-10 09:05:00+02:00,1.15484,2026-08-10 13:10:00+02:00,1.15649,2026-08-10 16:15:00+02:00,1.15515,2026-08-10 16:15:00+02:00,1.275687,49,0.000029
6055,6056,0.00112,down,2026-08-10 13:10:00+02:00,1.15649,2026-08-11 12:55:00+02:00,1.15318,2026-08-11 18:45:00+02:00,1.15462,2026-08-11 18:45:00+02:00,2.555454,282,0.000010
6056,6057,0.00112,up,2026-08-11 12:55:00+02:00,1.15318,2026-08-11 18:45:00+02:00,1.15462,2026-08-12 10:10:00+02:00,1.15332,2026-08-12 10:10:00+02:00,1.114929,70,0.000018
6057,6058,0.00112,down,2026-08-11 18:45:00+02:00,1.15462,2026-08-12 10:20:00+02:00,1.15323,2026-08-12 14:20:00+02:00,1.15470,2026-08-12 14:20:00+02:00,1.074874,187,0.000006
6058,6059,0.00112,up,2026-08-12 10:20:00+02:00,1.15323,2026-08-12 18:05:00+02:00,1.15606,2026-08-12 18:40:00+02:00,1.15462,2026-08-12 18:40:00+02:00,2.191051,93,0.000026
6059,6060,0.00112,down,2026-08-12 18:05:00+02:00,1.15606,2026-08-13 11:30:00+02:00,1.15121,2026-08-13 12:50:00+02:00,1.15269,2026-08-13 12:50:00+02:00,3.745789,209,0.000020


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [31]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [32]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,6060.000000,2.356606,1.423107,1.000001,1.143423,1.373014,1.902376,2.864330,4.143343,5.127939,7.616528,14.163872
t,6060.000000,39.757921,48.607817,1.000000,5.000000,10.000000,22.000000,49.000000,99.000000,145.000000,224.820000,547.000000
r,6060.000000,0.000190,0.000280,0.000006,0.000027,0.000051,0.000104,0.000217,0.000413,0.000632,0.001396,0.005896


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [33]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"3,030",2.370054,1.902476,40.15,22.00,1.905264e-04,1.046403e-04
down,"3,030",2.343158,1.902146,39.36,22.00,1.894025e-04,1.036665e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [34]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

# Single-threshold HMM

The HMM receives one observation per completed DC event: $\log(R)$. This remains a one-feature R model; the logarithm compresses R's strong right tail for the Gaussian emission baseline.

This is an **event-time** model. Its posterior updates only when a completed event becomes available at `information_time = dcc_idx`. Forward-filling a state across five-minute bars below is a visualization of the latest available belief, not bar-by-bar HMM inference.

The model is fitted on the first 80% of events. Filtering is causal in its observations, but filtered values inside the training interval are still in-sample because the fitted parameters use that complete interval. Only the final 20% is a genuinely held-out causal sequence.

## Prepare the single R observation and chronological split

In [35]:
HMM_DISTRIBUTION = DistType.NORMAL
N_HMM_STATES = 2
TRAIN_FRACTION = 0.80
HMM_RANDOM_STATE = 100
HMM_MAX_ITER = 500
HMM_N_RESTARTS = 5

hmm_events = events.copy()
hmm_events["log_r"] = log_r(hmm_events)

train_event_count = int(len(hmm_events) * TRAIN_FRACTION)
if train_event_count < 20 or len(hmm_events) - train_event_count < 2:
    raise ValueError("The chronological train/OOS split contains too few events.")

x_all = hmm_events[["log_r"]].to_numpy(dtype=np.float64)
x_train = x_all[:train_event_count]
oos_start_time = hmm_events.iloc[train_event_count]["information_time"]

print(f"Observation: log(R), shape={x_all.shape}")
print(f"Training events: {train_event_count:,}")
print(f"OOS events: {len(hmm_events) - train_event_count:,}")
print(f"First OOS information time: {oos_start_time}")

Observation: log(R), shape=(6060, 1)
Training events: 4,848
OOS events: 1,212
First OOS information time: 2025-10-29 16:25:00+02:00


## Fit and filter

State indices emitted by an HMM are arbitrary. After fitting, the state with the lower training-fitted emission mean is named `slow_R`; the other is named `fast_R`. This mapping uses fitted training parameters, not OOS outcomes.

In [36]:
hmm_model = create_simple_hmm_instance(
    dist_type=HMM_DISTRIBUTION,
    n_states=N_HMM_STATES,
    random_state=HMM_RANDOM_STATE,
    max_iter=HMM_MAX_ITER,
    inference_mode=InferenceMode.FILTERING,
    covariance_type="diag",
    min_cov=1e-6,
    n_restarts=HMM_N_RESTARTS,
)
hmm_model.fit(x_train)

filtered_probabilities = hmm_model.predict_proba(x_all)
if filtered_probabilities.shape != (len(hmm_events), N_HMM_STATES):
    raise ValueError(f"Unexpected probability shape: {filtered_probabilities.shape}")
np.testing.assert_allclose(filtered_probabilities.sum(axis=1), 1.0, rtol=1e-10, atol=1e-10)

emission_locations = np.array([
    float(np.ravel(parameters["means"])[0]) for parameters in hmm_model.parameters
])
raw_state_order = np.argsort(emission_locations)
state_names = np.array(["slow_R", "fast_R"])
raw_to_state_name = {int(raw_state): state_names[rank] for rank, raw_state in enumerate(raw_state_order)}
semantic_probabilities = filtered_probabilities[:, raw_state_order]
semantic_state_positions = np.argmax(semantic_probabilities, axis=1)

hmm_events["raw_state"] = np.argmax(filtered_probabilities, axis=1)
hmm_events["state_name"] = state_names[semantic_state_positions]
hmm_events["state_probability"] = semantic_probabilities.max(axis=1)
hmm_events["p_slow_r"] = semantic_probabilities[:, 0]
hmm_events["p_fast_r"] = semantic_probabilities[:, 1]
hmm_events["sample"] = "train"
hmm_events.iloc[train_event_count:, hmm_events.columns.get_loc("sample")] = "oos"

print(f"Inference mode: {hmm_model.inference_mode}")
print(f"Raw-state mapping: {raw_to_state_name}")

Inference mode: filtering
Raw-state mapping: {1: np.str_('slow_R'), 0: np.str_('fast_R')}


## Inspect what the states learned

The emission location is a mean in log-R space. Its exponential is a geometric characteristic R, not the arithmetic mean R. The transition matrix advances one completed DC event per step.

In [37]:
emission_summary = pd.DataFrame({
    "state_name": state_names,
    "raw_state": raw_state_order,
    "emission_mean_log_r": emission_locations[raw_state_order],
    "emission_geometric_r": np.exp(emission_locations[raw_state_order]),
})

semantic_transition_matrix = pd.DataFrame(
    hmm_model.transition_prob()[np.ix_(raw_state_order, raw_state_order)],
    index=pd.Index(state_names, name="from_state"),
    columns=pd.Index(state_names, name="to_state"),
)

state_event_summary = hmm_events.groupby(["sample", "state_name"], sort=False).agg(
    event_count=("event_id", "size"),
    mean_r=("r", "mean"),
    median_r=("r", "median"),
    mean_t=("t", "mean"),
    median_t=("t", "median"),
    mean_tmv=("tmv", "mean"),
    mean_probability=("state_probability", "mean"),
)

display(emission_summary.style.format(precision=6))
display(semantic_transition_matrix.style.format(precision=4))
display(state_event_summary.style.format(precision=6))

,state_name,raw_state,emission_mean_log_r,emission_geometric_r
0,slow_R,1,-9.530673,0.000073
1,fast_R,0,-8.037392,0.000323


to_state,slow_R,fast_R
from_state,,
slow_R,0.9207,0.0793
fast_R,0.1770,0.8230


## Map filtered event states onto price

Each update is indexed by its DCC information timestamp and forward-filled until the next completed event. The state therefore means **the latest event-time belief currently available**. It does not update between DCCs, and no state is assigned before the first completed event.

In [38]:
event_state_updates = hmm_events[[
    "information_time", "event_id", "state_name", "state_probability",
    "p_slow_r", "p_fast_r",
]].copy()
if event_state_updates["information_time"].duplicated().any():
    raise ValueError("Multiple HMM updates share an information timestamp.")

event_state_updates = event_state_updates.rename(columns={
    "information_time": "state_information_time",
    "event_id": "source_event_id",
}).set_index("state_information_time", drop=False)

bar_hmm_states = event_state_updates.reindex(prices.index).ffill()
first_information_time = event_state_updates.index[0]
assert bar_hmm_states.loc[bar_hmm_states.index < first_information_time, "state_name"].isna().all()

hmm_price_fig = plot_hmm_states_on_price(prices=prices, bar_states=bar_hmm_states, hmm_events=hmm_events,
                                         n_prices=20000, title=f"{SYMBOL} | theta={THETA:.4%} | filtered HMM on log(R)",
                                         oos_start_time=oos_start_time)
hmm_price_fig.show()

# HMM coherence audits

These audits test the specific contention raised by the price chart: do visually similar DC events receive different states for a defensible sequence-dependent reason, or is the model behaving like an incoherent rare-tail classifier?

No model is refitted here. All results use the same NORMAL HMM, log-R observation, chronological split, filtered probabilities, and state mapping created above.

In [39]:
audit_events = hmm_events.copy()
audit_events["previous_p_fast_r"] = audit_events["p_fast_r"].shift(1)
audit_events[[
    "event_id", "information_time", "sample", "r", "log_r",
    "previous_p_fast_r", "p_fast_r", "state_name",
]].tail()

,event_id,information_time,sample,r,log_r,previous_p_fast_r,p_fast_r,state_name
6055,6056,2026-08-11 18:45:00+02:00,oos,0.000010,-11.498104,0.000950,0.000036,slow_R
6056,6057,2026-08-12 10:10:00+02:00,oos,0.000018,-10.934131,0.000036,0.000214,slow_R
6057,6058,2026-08-12 14:20:00+02:00,oos,0.000006,-11.953331,0.000214,0.000008,slow_R
6058,6059,2026-08-12 18:40:00+02:00,oos,0.000026,-10.542645,0.000008,0.000705,slow_R
6059,6060,2026-08-13 12:50:00+02:00,oos,0.000020,-10.816128,0.000705,0.000310,slow_R


## Audit 1: log-R distributions by state

If the state names are meaningful, `fast_R` should occupy materially higher log-R values than `slow_R`. Overlap is expected because an HMM also uses the preceding belief and transition probabilities. Compare training and OOS rather than accepting in-sample separation alone.

In [40]:
log_r_audit_fig = plot_hmm_log_r_distributions(audit_events)
log_r_audit_fig.show()

## Audit 2: TMV–T geometry

Because $R = \mathrm{TMV}\,\theta/T$, equal R lies along a diagonal relationship in TMV–T space. Both axes are logarithmic. This chart reveals whether `fast_R` genuinely corresponds to high movement per unit time, even when two price-chart segments look similar.

In [41]:
tmv_t_audit_fig = plot_hmm_tmv_t_space(audit_events)
tmv_t_audit_fig.show()

## Audit 3: observation versus filtered posterior

The x-axis is the current event's log-R observation; the y-axis is the resulting filtered probability of `fast_R`. Hover `previous p(fast_R)` to see the sequence context entering the update. Vertical spread at nearly equal log-R is the direct footprint of the transition prior and previous belief.

In [42]:
probability_audit_fig = plot_hmm_probability_vs_log_r(audit_events)
probability_audit_fig.show()

## Audit 4: event-time state sequence

This removes bar-duration weighting entirely. Every completed DC event occupies one horizontal step, regardless of whether its state was later carried across five bars or five hundred bars on the price chart.

In [43]:
event_time_audit_fig = plot_hmm_event_time_strip(audit_events, train_event_count=train_event_count)
event_time_audit_fig.show()

## Audit 5: nearest OOS observations assigned to different states

Each row pairs an OOS `fast_R` event with the `slow_R` event having the nearest log-R. If current observations are almost identical but previous posteriors differ materially, the differing classification is genuine HMM sequence dependence. If neither the observation nor prior belief explains the difference, the fit deserves rejection.

In [44]:
oos_audit_events = audit_events.loc[audit_events["sample"].eq("oos")].copy()
slow_oos = oos_audit_events.loc[oos_audit_events["state_name"].eq("slow_R")].reset_index(drop=True)
fast_oos = oos_audit_events.loc[oos_audit_events["state_name"].eq("fast_R")].reset_index(drop=True)
if slow_oos.empty or fast_oos.empty:
    raise ValueError("Both HMM states must occur OOS for the nearest-pair audit.")

log_r_gaps = np.abs(
    fast_oos["log_r"].to_numpy()[:, None] - slow_oos["log_r"].to_numpy()[None, :]
)
nearest_slow_positions = log_r_gaps.argmin(axis=1)
pair_rows = []
for fast_position, slow_position in enumerate(nearest_slow_positions):
    fast_event = fast_oos.iloc[fast_position]
    slow_event = slow_oos.iloc[int(slow_position)]
    pair_rows.append({
        "log_r_gap": abs(float(fast_event["log_r"] - slow_event["log_r"])),
        "relative_r_gap": abs(float(fast_event["r"] - slow_event["r"]))
        / ((float(fast_event["r"]) + float(slow_event["r"])) / 2.0),
        "fast_event_id": int(fast_event["event_id"]),
        "slow_event_id": int(slow_event["event_id"]),
        "fast_time": fast_event["information_time"],
        "slow_time": slow_event["information_time"],
        "fast_r": fast_event["r"],
        "slow_r": slow_event["r"],
        "fast_t": int(fast_event["t"]),
        "slow_t": int(slow_event["t"]),
        "fast_tmv": fast_event["tmv"],
        "slow_tmv": slow_event["tmv"],
        "fast_previous_p": fast_event["previous_p_fast_r"],
        "slow_previous_p": slow_event["previous_p_fast_r"],
        "fast_current_p": fast_event["p_fast_r"],
        "slow_current_p": slow_event["p_fast_r"],
    })

nearest_opposite_state_pairs = pd.DataFrame(pair_rows).sort_values("log_r_gap").head(20)
nearest_opposite_state_pairs.style.format({
    "log_r_gap": "{:.6f}",
    "relative_r_gap": "{:.3%}",
    "fast_r": "{:.6e}",
    "slow_r": "{:.6e}",
    "fast_tmv": "{:.4f}",
    "slow_tmv": "{:.4f}",
    "fast_previous_p": "{:.4f}",
    "slow_previous_p": "{:.4f}",
    "fast_current_p": "{:.4f}",
    "slow_current_p": "{:.4f}",
})

,log_r_gap,relative_r_gap,fast_event_id,slow_event_id,fast_time,slow_time,fast_r,slow_r,fast_t,slow_t,fast_tmv,slow_tmv,fast_previous_p,slow_previous_p,fast_current_p,slow_current_p
78,0.000000,0.000%,5388,5500,2026-03-03 22:05:00+02:00,2026-03-18 14:20:00+02:00,9.767252e-05,9.767249e-05,12,25,1.0465,2.1802,0.9717,0.0017,0.5641,0.0272
70,0.000018,0.002%,5376,5989,2026-03-02 20:45:00+02:00,2026-07-16 19:35:00+02:00,2.359024e-04,2.359068e-04,17,5,3.5807,1.0532,0.7658,0.0029,0.8413,0.2027
112,0.000026,0.003%,5445,5467,2026-03-09 22:15:00+02:00,2026-03-11 16:55:00+02:00,1.561516e-04,1.561476e-04,23,10,3.2067,1.3942,0.9120,0.0251,0.7675,0.1029
90,0.000039,0.004%,5407,5936,2026-03-04 21:25:00+02:00,2026-06-30 15:35:00+02:00,1.429652e-04,1.429596e-04,8,13,1.0212,1.6594,0.8936,0.0312,0.7111,0.0883
150,0.000044,0.004%,5555,5278,2026-03-24 00:25:00+02:00,2026-02-09 03:45:00+02:00,1.691601e-04,1.691675e-04,29,7,4.3800,1.0573,0.8908,0.0005,0.7871,0.1003
43,0.000070,0.007%,5207,5322,2026-01-29 05:00:00+02:00,2026-02-19 19:00:00+02:00,8.743523e-05,8.742911e-05,51,62,3.9814,4.8398,0.9910,0.0002,0.5148,0.0202
154,0.000093,0.009%,5597,5844,2026-03-31 22:55:00+02:00,2026-06-08 08:55:00+02:00,2.618499e-04,2.618255e-04,13,5,3.0393,1.1689,0.4858,0.0094,0.7428,0.2572
95,0.000132,0.013%,5419,5652,2026-03-05 22:15:00+02:00,2026-04-10 20:20:00+02:00,1.903098e-04,1.902847e-04,14,6,2.3789,1.0194,0.6541,0.0096,0.6909,0.1396
115,0.000167,0.017%,5449,4893,2026-03-10 03:20:00+02:00,2025-11-10 13:10:00+02:00,1.199335e-04,1.199535e-04,26,21,2.7842,2.2491,0.9896,0.0176,0.7057,0.0524
163,0.000205,0.021%,5638,5879,2026-04-08 12:25:00+02:00,2026-06-15 04:55:00+02:00,3.111824e-04,3.112464e-04,7,14,1.9449,3.8906,0.8329,0.0003,0.9268,0.3208


# Audit 6: does the current state describe what happens next?

The previous audits explain the classification of event $n$. This audit moves the target forward by exactly one completed event. Every predictor below is available at `information_time_n`; every outcome belongs to event $n+1$ and is only used as a target.

The primary evidence is OOS. Train rows whose target crosses the train/OOS boundary are excluded. Pairs with non-increasing confirmation timestamps are also excluded because event $n+1$ would not be strictly later in physical time.

In [45]:
forward_events = audit_events.copy().reset_index(drop=True)
next_columns = {
    "event_id": "next_event_id",
    "information_time": "next_information_time",
    "sample": "next_sample",
    "state_name": "next_state_name",
    "r": "next_r",
    "log_r": "next_log_r",
    "t": "next_t",
    "tmv": "next_tmv",
    "direction": "next_direction",
}
for current_column, next_column in next_columns.items():
    forward_events[next_column] = forward_events[current_column].shift(-1)

information_positions = prices.index.searchsorted(
    pd.DatetimeIndex(forward_events["information_time"]), side="left"
)
forward_events["information_position"] = information_positions
forward_events["next_information_position"] = pd.Series(information_positions).shift(-1)
forward_events["next_confirmation_gap_bars"] = (
    forward_events["next_information_position"] - forward_events["information_position"]
)
forward_events["next_is_fast_r"] = forward_events["next_state_name"].eq("fast_R").astype(float)
forward_events.loc[forward_events["next_state_name"].isna(), "next_is_fast_r"] = np.nan

strictly_later = forward_events["next_information_time"] > forward_events["information_time"]
complete_target = forward_events[list(next_columns.values())].notna().all(axis=1)
forward_events["valid_forward_pair"] = strictly_later & complete_target
complete_forward_events = forward_events.loc[forward_events["valid_forward_pair"]].copy()

train_forward = complete_forward_events.loc[
    complete_forward_events["sample"].eq("train") & complete_forward_events["next_sample"].eq("train")
].copy()
oos_forward = complete_forward_events.loc[
    complete_forward_events["sample"].eq("oos") & complete_forward_events["next_sample"].eq("oos")
].copy()

assert (train_forward["next_information_time"] < oos_start_time).all()
assert (oos_forward["information_time"] >= oos_start_time).all()
assert (complete_forward_events["next_confirmation_gap_bars"] > 0).all()

excluded_non_later = int((complete_target & ~strictly_later).sum())
print(f"Valid train pairs: {len(train_forward):,}")
print(f"Valid OOS pairs: {len(oos_forward):,}")
print(f"Excluded non-later confirmation pairs: {excluded_non_later:,}")

Valid train pairs: 4,847
Valid OOS pairs: 1,211
Excluded non-later confirmation pairs: 0


In [46]:
oos_next_event_summary = oos_forward.groupby("state_name", sort=False).agg(
    current_event_count=("event_id", "size"),
    next_fast_rate=("next_is_fast_r", "mean"),
    next_r_mean=("next_r", "mean"),
    next_r_median=("next_r", "median"),
    next_r_p90=("next_r", lambda values: values.quantile(0.90)),
    next_t_mean=("next_t", "mean"),
    next_t_median=("next_t", "median"),
    next_tmv_mean=("next_tmv", "mean"),
    next_tmv_median=("next_tmv", "median"),
    next_confirmation_gap_bars_mean=("next_confirmation_gap_bars", "mean"),
    next_confirmation_gap_bars_median=("next_confirmation_gap_bars", "median"),
)
oos_state_transition_rates = pd.crosstab(
    oos_forward["state_name"],
    oos_forward["next_state_name"],
    normalize="index",
).rename_axis(index="current_state", columns="next_state")

display(oos_next_event_summary.style.format({
    "current_event_count": "{:,}",
    "next_fast_rate": "{:.2%}",
    "next_r_mean": "{:.6e}",
    "next_r_median": "{:.6e}",
    "next_r_p90": "{:.6e}",
    "next_t_mean": "{:.2f}",
    "next_t_median": "{:.1f}",
    "next_tmv_mean": "{:.4f}",
    "next_tmv_median": "{:.4f}",
    "next_confirmation_gap_bars_mean": "{:.2f}",
    "next_confirmation_gap_bars_median": "{:.1f}",
}))
display(oos_state_transition_rates.style.format("{:.2%}"))

,current_event_count,next_fast_rate,next_r_mean,next_r_median,next_r_p90,next_t_mean,next_t_median,next_tmv_mean,next_tmv_median,next_confirmation_gap_bars_mean,next_confirmation_gap_bars_median
state_name,,,,,,,,,,,
slow_R,"1,018",6.39%,1.147601e-04,6.736672e-05,2.254553e-04,53.32,34.0,2.3100,1.9026,52.65,34.0
fast_R,193,66.32%,2.403084e-04,1.638832e-04,5.548632e-04,22.69,12.0,2.1716,1.9352,26.27,15.0


next_state,fast_R,slow_R
current_state,,
fast_R,66.32%,33.68%
slow_R,6.39%,93.61%


In [47]:
next_event_outcomes_fig = plot_hmm_next_event_outcomes(complete_forward_events, sample="oos")
next_event_outcomes_fig.show()

## Does HMM history improve on current log(R)?

The next state is an internal HMM-defined target, so this is a coherence and incremental-information test—not evidence of trading value. Lower OOS Brier score and log loss are better; higher ROC AUC is better.

The key comparison is `current_log_R` versus `current_log_R_plus_p_fast`. The latter only deserves preference if the filtered probability—which carries prior-state information—improves OOS performance. The direct HMM forecast propagates the current filtered probabilities through the fitted transition matrix before event $n+1$ is observed.

In [48]:
class NextStateForecast(StrEnum):
    TRAIN_PREVALENCE = "train_prevalence"
    CURRENT_LOG_R = "current_log_R"
    HMM_TRANSITION = "hmm_transition"
    CURRENT_LOG_R_PLUS_P_FAST = "current_log_R_plus_p_fast"


semantic_transition_matrix = hmm_model.transition_prob()[np.ix_(raw_state_order, raw_state_order)]
forward_events["hmm_transition_p_next_fast"] = semantic_probabilities @ semantic_transition_matrix[:, 1]
complete_forward_events["hmm_transition_p_next_fast"] = forward_events.loc[
    complete_forward_events.index, "hmm_transition_p_next_fast"
]
train_forward = complete_forward_events.loc[train_forward.index].copy()
oos_forward = complete_forward_events.loc[oos_forward.index].copy()

train_target = train_forward["next_is_fast_r"].astype(int).to_numpy()
oos_target = oos_forward["next_is_fast_r"].astype(int).to_numpy()
if np.unique(train_target).size != 2 or np.unique(oos_target).size != 2:
    raise ValueError("Both next-event states must occur in train and OOS for forecast comparison.")

log_r_forecaster = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=HMM_RANDOM_STATE),
)
augmented_forecaster = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=HMM_RANDOM_STATE),
)
log_r_forecaster.fit(train_forward[["log_r"]], train_target)
augmented_forecaster.fit(train_forward[["log_r", "p_fast_r"]], train_target)

forecast_probabilities = {
    NextStateForecast.TRAIN_PREVALENCE: np.full(len(oos_forward), train_target.mean()),
    NextStateForecast.CURRENT_LOG_R: log_r_forecaster.predict_proba(oos_forward[["log_r"]])[:, 1],
    NextStateForecast.HMM_TRANSITION: oos_forward["hmm_transition_p_next_fast"].to_numpy(),
    NextStateForecast.CURRENT_LOG_R_PLUS_P_FAST: augmented_forecaster.predict_proba(
        oos_forward[["log_r", "p_fast_r"]]
    )[:, 1],
}

forecast_rows = []
metric_rows = []
for forecast_name, probabilities in forecast_probabilities.items():
    clipped_probabilities = np.clip(probabilities, 1e-12, 1.0 - 1e-12)
    metric_rows.append({
        "model": forecast_name.value,
        "brier_score": brier_score_loss(oos_target, clipped_probabilities),
        "log_loss": log_loss(oos_target, clipped_probabilities, labels=[0, 1]),
        "roc_auc": roc_auc_score(oos_target, clipped_probabilities),
        "mean_predicted_fast_rate": clipped_probabilities.mean(),
        "observed_fast_rate": oos_target.mean(),
    })
    forecast_rows.append(pd.DataFrame({
        "event_id": oos_forward["event_id"].to_numpy(),
        "model": forecast_name.value,
        "predicted_probability": clipped_probabilities,
        "actual_next_fast": oos_target,
    }))

next_state_forecast_metrics = pd.DataFrame(metric_rows).sort_values("log_loss")
next_state_forecasts = pd.concat(forecast_rows, ignore_index=True)
next_state_forecast_metrics.style.format({
    "brier_score": "{:.6f}",
    "log_loss": "{:.6f}",
    "roc_auc": "{:.4f}",
    "mean_predicted_fast_rate": "{:.2%}",
    "observed_fast_rate": "{:.2%}",
})

,model,brier_score,log_loss,roc_auc,mean_predicted_fast_rate,observed_fast_rate
3,current_log_R_plus_p_fast,0.079050,0.270899,0.8971,18.28%,15.94%
2,hmm_transition,0.080685,0.286856,0.9000,21.44%,15.94%
1,current_log_R,0.093719,0.312753,0.8739,19.69%,15.94%
0,train_prevalence,0.153053,0.490050,0.5000,29.75%,15.94%


In [49]:
calibration_rows = []
for model_name, model_forecasts in next_state_forecasts.groupby("model", sort=False):
    model_forecasts = model_forecasts.copy()
    if model_forecasts["predicted_probability"].nunique() == 1:
        model_forecasts["probability_bin"] = "constant"
    else:
        model_forecasts["probability_bin"] = pd.qcut(
            model_forecasts["predicted_probability"], q=10, duplicates="drop"
        ).astype(str)
    model_calibration = model_forecasts.groupby("probability_bin", observed=True).agg(
        mean_predicted_probability=("predicted_probability", "mean"),
        observed_fast_rate=("actual_next_fast", "mean"),
        event_count=("event_id", "size"),
    ).reset_index()
    model_calibration.insert(0, "model", model_name)
    calibration_rows.append(model_calibration)

next_state_calibration = pd.concat(calibration_rows, ignore_index=True)
next_state_calibration_fig = plot_hmm_next_state_calibration(next_state_calibration)
next_state_calibration_fig.show()

## How to interpret Audit 6

1. First compare the OOS next-event table and box plots. If the next-event distributions are almost identical after `slow_R` and `fast_R`, the current state is descriptive rather than forward-looking.
2. Compare `hmm_transition` with `train_prevalence`. Improvement shows sequence persistence has predictive content.
3. Compare `current_log_R_plus_p_fast` with `current_log_R`. Improvement is the stricter test that HMM history contributes beyond the current observation.
4. Inspect calibration, not only discrimination. A high ROC AUC with poor probability calibration is not a trustworthy probability forecast.
5. This remains an event-structure audit. It does not establish return predictability or trading utility.

# Audit 7: recent-event economic casebook

This casebook completes the qualitative step in the learning plan. It selects one recent, visually manageable OOS example of each filtered-state transition.

At the current DCC we know the confirmed extreme, completed-leg direction, TMV, T, R, filtered velocity state, and the new live DC mode. The following extreme, following DCC, following state, and post-confirmation excursion are shown only in hindsight. They are outcomes to inspect—not information the current decision could use.

In [50]:
class CasebookTransition(StrEnum):
    SLOW_TO_SLOW = "slow_R → slow_R"
    SLOW_TO_FAST = "slow_R → fast_R"
    FAST_TO_FAST = "fast_R → fast_R"
    FAST_TO_SLOW = "fast_R → slow_R"


CASEBOOK_MAX_LIFECYCLE_BARS = 800
CASEBOOK_PADDING_BARS = 12
TRANSITION_STATES = {
    CasebookTransition.SLOW_TO_SLOW: ("slow_R", "slow_R"),
    CasebookTransition.SLOW_TO_FAST: ("slow_R", "fast_R"),
    CasebookTransition.FAST_TO_FAST: ("fast_R", "fast_R"),
    CasebookTransition.FAST_TO_SLOW: ("fast_R", "slow_R"),
}

casebook_candidates = audit_events.copy().reset_index(drop=True)
casebook_candidates["previous_state_name"] = casebook_candidates["state_name"].shift(1)
casebook_candidates["previous_event_id"] = casebook_candidates["event_id"].shift(1)
casebook_next_columns = [
    "event_id", "state_name", "ext_end_idx", "ext_end_price", "ext_end_pos",
    "dcc_idx", "dcc_price", "dcc_pos", "tmv", "t", "r",
]
for column in casebook_next_columns:
    casebook_candidates[f"next_{column}"] = casebook_candidates[column].shift(-1)

casebook_candidates["live_mode"] = np.where(
    casebook_candidates["direction"].eq("up"), "down", "up"
)
casebook_candidates["confirmation_type"] = casebook_candidates["live_mode"] + "ward DCC"
casebook_candidates["lifecycle_bar_count"] = (
    casebook_candidates["next_dcc_pos"] - casebook_candidates["ext_start_pos"] + 1
)
casebook_candidates["post_confirmation_bars"] = (
    casebook_candidates["next_ext_end_pos"] - casebook_candidates["dcc_pos"]
)
casebook_candidates["post_confirmation_excursion_theta"] = np.where(
    casebook_candidates["live_mode"].eq("up"),
    (casebook_candidates["next_ext_end_price"] / casebook_candidates["dcc_price"] - 1.0) / THETA,
    (1.0 - casebook_candidates["next_ext_end_price"] / casebook_candidates["dcc_price"]) / THETA,
)

eligible_casebook_events = casebook_candidates.loc[
    casebook_candidates["sample"].eq("oos")
    & casebook_candidates["previous_state_name"].notna()
    & casebook_candidates["next_dcc_idx"].notna()
    & casebook_candidates["next_dcc_idx"].gt(casebook_candidates["dcc_idx"])
].copy()

selected_casebook_rows = []
missing_transitions = []
for transition, (previous_state, current_state) in TRANSITION_STATES.items():
    transition_candidates = eligible_casebook_events.loc[
        eligible_casebook_events["previous_state_name"].eq(previous_state)
        & eligible_casebook_events["state_name"].eq(current_state)
    ]
    if transition_candidates.empty:
        missing_transitions.append(transition.value)
        continue
    manageable_candidates = transition_candidates.loc[
        transition_candidates["lifecycle_bar_count"].le(CASEBOOK_MAX_LIFECYCLE_BARS)
    ]
    selection_pool = manageable_candidates if not manageable_candidates.empty else transition_candidates
    selected_event = selection_pool.iloc[-1].copy()
    selected_event["transition_label"] = transition.value
    selected_casebook_rows.append(selected_event)

if missing_transitions:
    raise ValueError(f"No eligible recent OOS examples for transitions: {missing_transitions}")

casebook_events = pd.DataFrame(selected_casebook_rows).reset_index(drop=True)
casebook_events["window_start_pos"] = (
    casebook_events["ext_start_pos"] - CASEBOOK_PADDING_BARS
).clip(lower=0).astype(int)
casebook_events["window_end_pos"] = (
    casebook_events["next_dcc_pos"] + CASEBOOK_PADDING_BARS
).clip(upper=len(prices) - 1).astype(int)
casebook_events["window_start"] = prices.index[casebook_events["window_start_pos"]]
casebook_events["window_end"] = prices.index[casebook_events["window_end_pos"]]

assert (casebook_events["dcc_idx"] == casebook_events["information_time"]).all()
assert (casebook_events["next_dcc_idx"] > casebook_events["dcc_idx"]).all()
casebook_events[["transition_label", "event_id", "information_time", "lifecycle_bar_count"]]

,transition_label,event_id,information_time,lifecycle_bar_count
0,slow_R → slow_R,6059,2026-08-12 18:40:00+02:00,319.0
1,slow_R → fast_R,6036,2026-07-31 18:50:00+02:00,118.0
2,fast_R → fast_R,6020,2026-07-30 01:30:00+02:00,62.0
3,fast_R → slow_R,6037,2026-08-03 04:05:00+02:00,132.0


In [51]:
casebook_ledger_columns = [
    "transition_label", "event_id", "information_time", "direction", "confirmation_type",
    "live_mode", "state_name", "tmv", "t", "r", "p_fast_r", "state_probability",
    "post_confirmation_excursion_theta", "post_confirmation_bars", "next_event_id",
    "next_state_name", "next_tmv", "next_t", "next_r", "next_dcc_idx",
]
casebook_ledger = casebook_events[casebook_ledger_columns].copy()
casebook_ledger.style.format({
    "event_id": "{:.0f}",
    "tmv": "{:.4f}",
    "t": "{:.0f}",
    "r": "{:.6e}",
    "p_fast_r": "{:.3f}",
    "state_probability": "{:.3f}",
    "post_confirmation_excursion_theta": "{:.2f}θ",
    "post_confirmation_bars": "{:.0f}",
    "next_event_id": "{:.0f}",
    "next_tmv": "{:.4f}",
    "next_t": "{:.0f}",
    "next_r": "{:.6e}",
})

,transition_label,event_id,information_time,direction,confirmation_type,live_mode,state_name,tmv,t,r,p_fast_r,state_probability,post_confirmation_excursion_theta,post_confirmation_bars,next_event_id,next_state_name,next_tmv,next_t,next_r,next_dcc_idx
0,slow_R → slow_R,6059,2026-08-12 18:40:00+02:00,up,downward DCC,down,slow_R,2.1911,93,2.638685e-05,0.001,0.999,2.64θ,202,6060,slow_R,3.7458,209,2.007313e-05,2026-08-13 12:50:00+02:00
1,slow_R → fast_R,6036,2026-07-31 18:50:00+02:00,down,upward DCC,up,fast_R,3.4696,3,1.295314e-03,0.947,0.947,6.70θ,100,6037,slow_R,7.7689,103,8.447686e-05,2026-08-03 04:05:00+02:00
2,fast_R → fast_R,6020,2026-07-30 01:30:00+02:00,down,upward DCC,up,fast_R,1.2617,8,1.766415e-04,0.825,0.825,0.89θ,30,6021,slow_R,2.0669,37,6.256420e-05,2026-07-30 05:25:00+02:00
3,fast_R → slow_R,6037,2026-08-03 04:05:00+02:00,up,downward DCC,down,slow_R,7.7689,103,8.447686e-05,0.441,0.559,0.61θ,13,6038,slow_R,1.6609,24,7.750764e-05,2026-08-03 05:30:00+02:00


In [52]:
recent_casebook_fig = plot_hmm_recent_event_casebook(prices, casebook_events)
recent_casebook_fig.show()

## Reading each case without look-ahead

At the colored triangular DCC marker, say the following out loud:

1. **What ended?** The green or red extreme-to-extreme segment is the completed leg. Its direction is retrospective.
2. **What is the mode now?** The opposite DCC has switched the live DC mode. A completed up leg produces a downward DCC and live down mode; a completed down leg produces an upward DCC and live up mode.
3. **How quickly did the completed leg develop?** TMV, T, and R determine whether its normalized velocity state is `slow_R` or `fast_R`.
4. **What does the shading mean?** It carries that completed-event velocity state forward until the next DCC. It is not a fresh classification on every bar.
5. **What was unknown?** The purple next extreme and black next DCC were unknowable at the current DCC. Use them only to judge whether the state was followed by economically different behaviour.

A precise market description at the current marker is: *The prior [up/down] leg completed with [slow/fast] normalized velocity; the opposite DCC has now put the process into [down/up] mode.*